# mmpartnet demo: modular data layer -> frozen PARNET -> profile recovery

Minimal end-to-end run on **public** data, built to be the shared skeleton the team fills out.
It shows the three moving parts we want to keep swappable:

1. a **DataSource** (here `encode_bigwig`: public ENCODE eCLIP, read remotely, no multi-GB download),
2. **format preprocessing** in one place (`preprocess.to_profile`, selected by `cfg.target`),
3. an **experiment** that does not care where the data came from (frozen pretrained PARNET, profile Pearson/Spearman vs nulls).

Switch backend by name (`get_source("encode_bam_counts", cfg)`) or format by `cfg.target`; nothing
downstream changes. See `docs/DATA_INVENTORY.md` for what each source is established vs a surrogate of.
Runs on a a GPU node via `scripts/run_demos.sh` (nbconvert, headless).

## Definitions

The model emits per-position logits $z\in\mathbb{R}^{L}$ over the $L=600$ window positions; a **softmax**
turns them into a categorical **profile** over positions,
$$\hat p_i=\operatorname{softmax}(z)_i=\frac{e^{z_i}}{\sum_{j=1}^{L}e^{z_j}},\qquad \sum_{i=1}^{L}\hat p_i=1 .$$
We score the predicted profile $\hat p$ against the observed, normalized eCLIP profile $o$ with the
**Pearson** correlation
$$r(\hat p,o)=\frac{\sum_i(\hat p_i-\bar{\hat p})(o_i-\bar o)}{\sqrt{\sum_i(\hat p_i-\bar{\hat p})^2}\ \sqrt{\sum_i(o_i-\bar o)^2}},$$
and **Spearman** $\rho$, i.e. the same formula on the rank-transformed vectors. Higher $r$ means the model
puts probability where the reads actually are.

In [ ]:
import os, sys, pathlib
import numpy as np

# put repo src/ on the path whether we launch from notebooks/demo/ or the repo root
_here = pathlib.Path.cwd().resolve()
REPO = next((c for c in (_here, *_here.parents) if (c / "src" / "mmpartnet").is_dir()), _here)
sys.path.insert(0, str(REPO / "src"))

from mmpartnet import config
from mmpartnet.data import DataConfig, get_source, iter_records, list_sources, preprocess

print("repo        :", REPO)
print("data sources:", list_sources())
print("target kinds:", preprocess.list_kinds())
for label, p in [("DATA", config.DATA), ("HG38", config.HG38), ("PARNET wts", config.PARNET_WEIGHTS)]:
    print(f"  {label:11}: exists={p.exists()}  {p}")

## 1. Pick a source + a data slice

`DataConfig` is the whole spec for one run. Env vars (`MMP_SOURCE`, `MMP_GROUP`, `MMP_CELL`, `MMP_NWIN`)
override the defaults so the GPU job can scale the slice up without editing the notebook. `group`
resolves through `io.groups` (a curated set, an ATtRACT family, or a `SYM,SYM,...` list).

In [ ]:
SOURCE = os.environ.get("MMP_SOURCE", "encode_bigwig")
GROUP  = os.environ.get("MMP_GROUP", "QKI,PTBP1,IGF2BP1")
CELL   = os.environ.get("MMP_CELL", "HepG2")
NWIN   = int(os.environ.get("MMP_NWIN", "6"))

cfg = DataConfig(source=SOURCE, group=GROUP, cell=CELL, nwin=NWIN, target="density")
src = get_source(SOURCE, cfg)
print(f"source={SOURCE}  group={GROUP}  cell={CELL}  nwin={NWIN}  target={cfg.target}")
print("RBPs in this slice:", src.rbps())

## 2. Stream uniform records

`iter_records` pulls each window's sequence + raw observed target from the source, applies the
`cfg.target` preprocessing (`density` here: abs + normalize to a position-distribution), and keeps up to
`nwin` windows per RBP that pass the `>= min_sum` read filter. The experiment below only sees
`{rbp, window, sequence, target}`, i.e. it is source-agnostic.

In [ ]:
records = list(iter_records(src))
by_rbp = {}
for r in records:
    by_rbp[r["rbp"]] = by_rbp.get(r["rbp"], 0) + 1
print(f"{len(records)} windows passed the >= {cfg.min_sum:.0f}-read filter")
print("per-RBP:", by_rbp)
assert records, "no records; check data paths (config cell above) and network access to ENCODE"

## 3. Frozen pretrained PARNET, profile recovery

This reproduces the PARNET demo's **evaluation** (per-RBP profile Pearson/Spearman on windows with
>=10 reads) on our public data: predict the per-nt profile with the frozen model, score it against the
observed eCLIP profile, and against two established nulls for autocorrelated genomic signal:
a circular shift (preserves autocorrelation) and a center-bump baseline (peak-centered windows enrich the
middle by construction). Real binding shape must beat both, not just a trivial i.i.d. shuffle.

In [ ]:
import torch
from mmpartnet.process.onehot import batch_onehot
from mmpartnet.models.parnet import load_parnet

def pearson(x, y):
    x = x - x.mean(); y = y - y.mean()
    d = np.sqrt((x * x).sum() * (y * y).sum())
    return float((x * y).sum() / d) if d > 1e-9 else np.nan

def spearman(x, y):
    rx = np.argsort(np.argsort(x)).astype(float)
    ry = np.argsort(np.argsort(y)).astype(float)
    return pearson(rx, ry)

def center_bump(L, sigma=None):
    sigma = sigma or L / 12.0
    x = np.arange(L, dtype=float)
    b = np.exp(-0.5 * ((x - (L - 1) / 2.0) / sigma) ** 2)
    return b / b.sum()

m = load_parnet()
cb = center_bump(cfg.lwin)
rows = []
for r in records:
    tr = m.track_index(r["rbp"], CELL)
    if tr is None:
        continue
    x = batch_onehot([r["sequence"]], device=m.device)
    pred = m.full(x)["total"][0, tr].detach().cpu().numpy()
    obs = r["target"]
    rng = np.random.default_rng(len(rows))
    k = int(rng.integers(cfg.lwin // 8, cfg.lwin - cfg.lwin // 8))
    rows.append(dict(rbp=r["rbp"], pred=pred, obs=obs,
                     pearson=pearson(pred, obs), spearman=spearman(pred, obs),
                     circ=pearson(pred, np.roll(obs, k)), cbump=pearson(cb, obs)))
print(f"scored {len(rows)} windows on device={m.device}")

In [ ]:
P = np.array([r["pearson"] for r in rows]); S = np.array([r["spearman"] for r in rows])
C = np.array([r["circ"] for r in rows]);    B = np.array([r["cbump"] for r in rows])
mp, mc, mb = float(np.nanmean(P)), float(np.nanmean(C)), float(np.nanmean(B))
print(f"mean Pearson    = {mp:+.3f}")
print(f"mean Spearman   = {float(np.nanmean(S)):+.3f}")
print(f"null circular   = {mc:+.3f}   (autocorrelation-matched)")
print(f"null center-bump= {mb:+.3f}   (peak-centring baseline)")
real = (mp - mc > 0.05) and (mp - mb > 0.05)
print(f"verdict: Pearson-circular={mp-mc:+.3f}, Pearson-centerbump={mp-mb:+.3f} -> "
      + ("REAL SHAPE (beats established nulls)" if real
         else "NOT BEYOND center/AUTOCORR (the density proxy is center-confounded; see note below)"))

**Reading the verdict.** On the `density` target (ENCODE RPM read-density, a **surrogate** for the
established single-nt 5' crosslink counts) the recovery is largely center-confounded, so the honest
verdict here is usually *not beyond center/autocorrelation*. On the established **counts** target
(`cfg.target="counts"` once the `encode_bam_counts` source is filled) the frozen model genuinely beats
the nulls (mean Pearson ~+0.29 in our verification run). That gap is exactly why the counts source is the
established swap; the density path is the runs-now scaffold. Details + per-surrogate validity in
`docs/DATA_INVENTORY.md`.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

best = max(rows, key=lambda r: r["pearson"])
xx = np.arange(cfg.lwin)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(xx, best["obs"],  lw=1.0, label="observed (ENCODE density)")
ax.plot(xx, best["pred"], lw=1.0, label="PARNET predicted")
ax.set_title(f"{best['rbp']} best window   Pearson={best['pearson']:+.3f}")
ax.set_xlabel("position in 600-nt window"); ax.set_ylabel("profile (prob)")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
out = REPO / "notebooks" / "demo" / "demo_profile.png"
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=110)
print("saved", out)

## 4. Switch the backend (the point of the scaffold)

```python
# established target instead of the density surrogate (fill encode_bam_counts.observed first):
cfg = DataConfig(source="encode_bam_counts", group=GROUP, cell=CELL, nwin=NWIN, target="counts")
src = get_source("encode_bam_counts", cfg)

# lab canonical substrate when it lands:
cfg = DataConfig(source="hfds", group=GROUP, target="hfds", extra={"path": "<dataset>"})
```

Nothing in section 3 changes. To add a format, register one function in `preprocess.py`; to add a source,
drop a module in `data/sources/` and import it in `data/sources/__init__.py`. Stubs there
(`encode_bam_counts`, `hfds`, `local_pt`) carry fill instructions. This is milestone-1 plumbing: a
swappable data layer + a reproducible frozen-PARNET baseline we can grow into the multimodal model.